# Validation: does the model return sensible comparables?

Every number in the README's validation section comes from this notebook. The
checks run cheapest-first: if feature resolution or the name join is broken,
nothing after it means anything.

Run the pipeline once before opening it:

```bash
python run_pipeline.py --stage all
```

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src import config, paths, similarity
from src.name_matching import normalise_name

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

features = pd.read_parquet(paths.FEATURES)
percentiles = pd.read_parquet(paths.PERCENTILES)
comps = pd.read_parquet(paths.SIMILARITY)
values = pd.read_parquet(paths.VALUE_PREDICTIONS)
models = joblib.load(paths.SIMILARITY_MODELS)
seasons = dict(zip(features["player_season_id"], features["season"].astype(str)))

# Accent-insensitive name search: "Luis Diaz" must find FBref's "Luis Díaz".
features["name_key"] = features["player"].map(normalise_name)


def find(name, season=None):
    """Rows for a player, ignoring accents and case.

    An exact name match wins; only if there is none does it fall back to a
    substring match, so "Rodri" finds Rodri rather than Rodrigo Ely.
    """
    key = normalise_name(name)
    rows = features[features["name_key"] == key]
    if rows.empty:
        rows = features[features["name_key"].str.contains(key, regex=False)]
    if season is not None:
        rows = rows[rows["season"].astype(str) == str(season)]
    return rows


print(f"{len(features):,} player-seasons in the model")
print(features.pivot_table(index="league", columns="season", values="player", aggfunc="size", fill_value=0))
print()
print(features.groupby("position_group").size().sort_values().to_string())

## 1. Did every feature resolve?

FBref renames table headers from time to time. A feature that fails to resolve
is dropped from every group that uses it, which weakens the profiles without
raising an error, so the resolution report is checked first.

In [ ]:
resolution = pd.read_csv(paths.FEATURE_RESOLUTION)
missing = resolution[resolution["status"] == "MISSING"]

print(f"{len(resolution) - len(missing)}/{len(resolution)} features resolved")
if len(missing):
    print("\nUnresolved (fix the patterns in src/config.py against reports/fbref_columns.txt):")
    display(missing[["feature", "patterns_tried"]])

## 2. What did the name join achieve?

FBref and Transfermarkt are joined on name and birth year, then every match is
confirmed against the club the player actually appeared for that season. A low
match rate would silently remove exactly the recognisable players a case study
needs, so the rate is reported rather than assumed.

In [ ]:
summary = json.loads(paths.MATCH_SUMMARY.read_text())
print(f"Name match rate: {summary['match_rate']:.1%} of {summary['total']:,} distinct players")
print(summary["decisions"])

player_seasons = pd.read_parquet(paths.PLAYER_SEASONS, columns=["club_check"])
print("\nClub check:", player_seasons["club_check"].value_counts().to_dict())

print(f"\nModel rows with a market value: {features['market_value_eur'].notna().mean():.1%}")
print(f"Model rows with a Transfermarkt position: {(features['position_source'] == 'transfermarkt').mean():.1%}")
print(f"Model rows with a dated contract: {features['contract_years_remaining'].notna().mean():.1%}")

In [ ]:
# Spot-check that well-known players survived the join with a value attached.
WATCHLIST = ["Alexis Mac Allister", "Bruno Guimaraes", "Milos Kerkez", "Nico Williams", "Joshua Kimmich"]

for name in WATCHLIST:
    rows = find(name)
    if rows.empty:
        print(f"MISSING  {name}")
        continue
    row = rows.nlargest(1, "minutes").iloc[0]
    flag = "" if pd.notna(row["market_value_eur"]) else "   <-- no market value joined"
    print(f"ok       {name:24s} {row['position_group']:3s} {row['team']}{flag}")

## 3. Style dimensions

The top and bottom loadings of the first three components per group. These are
what the README's style-dimension table names in football terms.

In [ ]:
loadings = pd.read_csv(paths.PCA_LOADINGS)


def describe(group, n=4):
    chunk = loadings[loadings["position_group"] == group]
    for component in chunk["component"].unique()[:3]:
        block = chunk[chunk["component"] == component].sort_values("loading", ascending=False)
        variance = block["explained_variance_ratio"].iloc[0]
        print(f"\n{group} {component}  ({variance:.0%} of variance)")
        print("  high:", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.head(n).itertuples()))
        print("  low: ", ", ".join(f"{r.feature} {r.loading:+.2f}" for r in block.tail(n).itertuples()))


for group in config.POSITION_GROUPS:
    describe(group)

## 4. The eye test

Top five comparables for players whose game is well known. What failure looks
like: a target man returning false nines (the features measure output, not
role), or a deep-lying playmaker returning every high-volume passer (they lack
the zone and progression detail that separates a regista from a metronome).

In [ ]:
def comps_for(name, season=None, n=5):
    pool = find(name, season)
    if pool.empty:
        print(f"Not found: {name}")
        return None
    row = pool.nlargest(1, "minutes").iloc[0]
    table = similarity.top_comps(row["player_season_id"], comps, n=n)
    out = table.merge(
        features[["player_season_id", "player", "team", "season", "age", "market_value_eur"]]
        .rename(columns={"player_season_id": "comp_player_season_id"}),
        on="comp_player_season_id",
    )
    print(f"\n=== {row['player']} ({row['team']}, {row['season']}, {row['position_group']}) ===")
    return out[["rank", "player", "team", "season", "age", "similarity", "market_value_eur"]]


EYE_TEST = [
    ("Rodri", "2223"),             # DM - should return controllers, not destroyers
    ("Virgil van Dijk", "2425"),   # CB - should return ball-playing centre-backs
    ("Mohamed Salah", "2425"),     # W  - should return high-volume shooters
    ("Erling Haaland", "2425"),    # ST - should NOT return false nines
    ("Alexander-Arnold", "2425"),  # FB - should return creative, inverted full-backs
]

for name, season in EYE_TEST:
    table = comps_for(name, season)
    if table is not None:
        display(table)

**Verdicts (draft - a starting point; rewrite in your own words after reading the tables):**

- Rodri - Partey, Bentaleb, Busquets, Santamaria, Højbjerg: deep controllers rather
  than pure ball-winners, with Busquets the eye-catching one. _Hit?_
- Virgil van Dijk - Tah, Alexsandro, Rrahmani, de Vrij: composed centre-backs from
  possession sides. _Hit?_
- Mohamed Salah - Mané, Martinelli, Greenwood, Lookman, Barcola: wide forwards who
  shoot and carry. _Hit?_
- Erling Haaland - Moukoko, Sørloth, Watkins, Lewandowski: penalty-box strikers, no
  false nines. _Hit?_
- Trent Alexander-Arnold - Trippier, Frankowski, Alba, Biraghi, Porro: crossers and
  set-piece takers. _Likely the miss: the full-back features see crossing volume
  and chance creation, but nothing says WHERE the passes come from, so a full-back
  who inverts into midfield to play long switches looks like a byline crosser._

## 5. Self-consistency

The same player a season later is the best comparable there is. For every
player with two seasons in the same group, `similarity.own_season_ranks` ranks
everyone in the group against him and records where his own other season
lands - computed exactly from the PCA scores, so a season ranked outside the
stored neighbour list counts as a miss instead of disappearing. A random
embedding would place it around the middle of the group (50%).

In [ ]:
ranks = similarity.own_season_ranks(models, seasons)
ranks["rank_pct"] = ranks["rank"] / ranks["group_size"]

print(json.dumps(json.loads(paths.VALIDATION.read_text()), indent=2))

summary_by_gap = ranks.groupby("season_gap").agg(
    scored=("rank", "size"),
    median_rank=("rank", "median"),
    median_rank_pct=("rank_pct", "median"),
    top_5pct_share=("rank_pct", lambda s: (s <= 0.05).mean()),
)
display(summary_by_gap.round(3))

ax = ranks.pivot(columns="season_gap", values="rank_pct").plot(
    kind="hist", bins=40, alpha=0.6, figsize=(7, 3),
    title="Own other season, as a share of the group (lower is better)",
)
ax.set_xlabel("rank / group size")

## 6. Minutes sensitivity

The model includes everyone above 600 minutes and the app defaults its slider
to 900. How much do the top-5 comparables change between the two pools? High
churn would mean low-minutes players are adding noise.

In [ ]:
TIGHT = config.MIN_MINUTES                # 900
LOOSE = config.MIN_MINUTES_SENSITIVITY    # 600

tight_ids = set(features.loc[features["minutes"] >= TIGHT, "player_season_id"])
others = comps[~comps["same_player"]]

overlaps = []
for player_season_id in sorted(tight_ids)[:400]:
    mine = others[others["player_season_id"] == player_season_id]
    loose_top5 = set(mine.nsmallest(5, "rank")["comp_player_season_id"])
    tight_top5 = set(mine[mine["comp_player_season_id"].isin(tight_ids)].nsmallest(5, "rank")["comp_player_season_id"])
    overlaps.append(len(loose_top5 & tight_top5))

print(f"Mean overlap of top-5 comparables, {LOOSE}+ pool vs {TIGHT}+ pool: {np.mean(overlaps):.2f} / 5")

## 7. Is the value model learning more than league and age?

The characteristic failure: an undervalued shortlist that is all 33-year-olds
in the cheapest league, because the model learnt league and age and nothing
else. Compare the flagged players with the pool.

In [ ]:
metrics = json.loads(paths.VALUE_METRICS.read_text())
display(pd.DataFrame({k: v for k, v in metrics.items() if k != "selected_model"}).round(3))
print("selected:", metrics["selected_model"])

flagged = values[values["undervalued"]]
display(pd.DataFrame({
    "pool": values["league"].value_counts(normalize=True),
    "flagged": flagged["league"].value_counts(normalize=True),
}).round(3))
print("Mean log residual by league (close to 0 = no league bias):")
print(values.groupby("league")["log_residual"].mean().round(3).to_string())

### Backtest: did the 2022-23 flags rise in value?

The test of the value flag that does not rely on the model's own numbers. For
every 2022-23 player-season, compare the market value used in the model with
the player's last Transfermarkt valuation on or before 1 July 2025, two
seasons later. Split by age, because young players rise anyway.

Two caveats: only players still valued in 2025 are counted, and part of what
the flag catches is Transfermarkt lagging behind breakthrough seasons - which
is itself useful to a recruiter, but is not the same as the model knowing
something the market cannot.

In [ ]:
valuations = pd.read_parquet(paths.TM_VALUATIONS).sort_values("date")
later = valuations[valuations["date"] <= "2025-07-01"].groupby("player_id").last()

cohort = values[(values["season"].astype(str) == "2223") & values["tm_player_id"].notna()].copy()
cohort["value_2025"] = cohort["tm_player_id"].map(later["market_value_in_eur"])
cohort["valued_on"] = cohort["tm_player_id"].map(later["date"])
cohort = cohort[cohort["valued_on"] >= "2025-01-01"]
cohort["growth"] = cohort["value_2025"] / cohort["market_value_eur"]
cohort["age_band"] = pd.cut(cohort["age"], [0, 22, 26, 29, 50], labels=["<=22", "23-26", "27-29", "30+"])

backtest = cohort.groupby(["age_band", "undervalued"], observed=True)["growth"].agg(
    players="size", median_growth="median", share_rose=lambda g: (g > 1).mean(),
)
display(backtest.round(2))
display(cohort.groupby("undervalued")["growth"].agg(
    players="size", median_growth="median", share_rose=lambda g: (g > 1).mean()).round(2))

## 8. Known transfers

Real replacement signings: would the model have surfaced the replacement as a
comparable to the player he replaced? The replacement's season *before* the
move is the fair test; his season at the new club is shown too, because ranking
high there means he went on to play the departed player's role.

In [ ]:
# (departed, his last season before leaving, replacement, replacement's season before joining)
TRANSFER_PAIRS = [
    ("Granit Xhaka", "2223", "Declan Rice", "2223"),       # Arsenal 2023
    ("Gundogan", "2223", "Kovacic", "2223"),               # Manchester City 2023
    ("Fabinho", "2223", "Wataru Endo", "2223"),            # Liverpool 2023
    ("Karim Benzema", "2223", "Joselu", "2223"),           # Real Madrid 2023
    ("Alexander-Arnold", "2425", "Frimpong", "2425"),      # Liverpool 2025
    ("Thomas Partey", "2425", "Zubimendi", "2425"),        # Arsenal 2025
    ("De Bruyne", "2425", "Reijnders", "2425"),            # Manchester City 2025
]


def rank_in_group(departed_id, candidate_ids):
    """Exact rank (1 = closest) of each candidate among the departed player's group."""
    group = features.loc[features["player_season_id"] == departed_id, "position_group"].iloc[0]
    model = models[group]
    ids = list(model["ids"])
    unit = model["scores"] / np.linalg.norm(model["scores"], axis=1, keepdims=True)
    sims = unit @ unit[ids.index(departed_id)]
    sims[ids.index(departed_id)] = -np.inf
    out = {}
    for cid in candidate_ids:
        out[cid] = int((sims > sims[ids.index(cid)]).sum()) + 1 if cid in ids else None
    return group, len(ids), out


rows = []
for departed, d_season, replacement, r_season in TRANSFER_PAIRS:
    d = find(departed, d_season)
    if d.empty:
        rows.append({"departed": departed, "replacement": replacement, "result": "departed not in sample"})
        continue
    d = d.nlargest(1, "minutes").iloc[0]
    r = find(replacement)
    group, size, ranked = rank_in_group(d["player_season_id"], list(r["player_season_id"]))
    before = r[r["season"].astype(str) == r_season]
    before_rank = ranked.get(before["player_season_id"].iloc[0]) if len(before) else None
    best_id = min((k for k, v in ranked.items() if v), key=ranked.get, default=None)
    rows.append({
        "departed": f"{d['player']} ({d['season']}, {group})",
        "replacement": replacement,
        "replacement groups": ", ".join(sorted(set(r["position_group"]))),
        "rank before move": before_rank,
        "best rank, any season": f"{ranked[best_id]} ({seasons[best_id]})" if best_id else None,
        "group size": size,
    })

display(pd.DataFrame(rows))

---

## Numbers used in the README

In [ ]:
validation = json.loads(paths.VALIDATION.read_text())
gap1 = validation["by_season_gap"]["1"]
ridge = metrics[metrics["selected_model"]]
print(f"Player-seasons in the model:          {len(features):,}")
print(f"Name match rate:                      {summary['match_rate']:.1%}")
print(f"Own adjacent season, median rank:     {gap1['median_rank']:.0f} (top {gap1['median_rank_pct']:.1%} of the group)")
print(f"Own adjacent season in top 5% of group: {gap1['top_5pct_share']:.1%}")
print(f"Value model median abs. error (CV):   {ridge['median_absolute_percentage_error']:.1f}%")
print(f"Smallest position group:              {features['position_group'].value_counts().idxmin()} "
      f"({features['position_group'].value_counts().min()} player-seasons)")